# Gemma 4 12B LoRA: inference and evaluation

Loads the trained adapter and scores one split. Set `SPLIT` below.
Training lives in `gemma4_lora_finetune.ipynb`; nothing here allocates
optimizer or gradient state, which is what keeps the model inside VRAM.

Prompt construction is imported from `prompts.py` so that it cannot drift
from what the model was trained on.

In [1]:
import os, json, pathlib
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from metrics import evaluate_predictions
from prompts import LEVELS, build_messages

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

LABEL2ID = {lvl: i for i, lvl in enumerate(LEVELS)}
NUM_CLASSES = len(LEVELS)

SPLIT = "val"          # "val" while developing, "test" for the final run


@dataclass
class CFG:
    adapter_path: str = "runs/gemma4_lora/lora_adapter"
    data_dir: str = "data"
    text_col: str = "text"
    label_col: str = "label"
    max_seq_length: int = 3072
    load_in_4bit: bool = True
    text_only: bool = False         # True crashes on Gemma 4; 12B is encoder-free anyway
    empty_cache_every: int = 50
    run_dir: str = "runs/gemma4_lora"


cfg = CFG()
os.makedirs(cfg.run_dir, exist_ok=True)

## Data

In [2]:
df = pd.read_json(f"{cfg.data_dir}/{SPLIT}.jsonl", lines=True)
df = df[df[cfg.label_col].isin(LEVELS)].reset_index(drop=True)
print(f"{SPLIT}: {len(df)} texts")
print(df[cfg.label_col].value_counts().sort_index())

val: 599 texts
label
A2      30
A2+    136
B1     128
B1+     95
B2     110
B2+     49
C1      37
C1+     14
Name: count, dtype: int64


## Load the adapter

Unsloth resolves the base model from `adapter_config.json` and pulls it from
the local HF cache, so there is no download.

In [3]:
# %%
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg.adapter_path,
    max_seq_length=cfg.max_seq_length,
    dtype=None,
    load_in_4bit=cfg.load_in_4bit,
)
FastLanguageModel.for_inference(model)
tok = tokenizer.tokenizer


def vram():
    return (torch.cuda.memory_allocated() / 1e9,
            torch.cuda.memory_reserved() / 1e9)


print("allocated %.1f GB | reserved %.1f GB" % vram())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0912 08:50:06.200000 26396 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0912 08:50:06.257000 26396 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


c:\Users\coope\AppData\Local\Programs\Python\Python312\Lib\site-packages\unsloth\import_fixes.py:2124: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


==((====))==  Unsloth 2026.9.4: Fast Gemma4_Unified patching. Transformers: 5.17.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.842 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

allocated 8.3 GB | reserved 8.4 GB


Check Task Manager now: dedicated GPU memory should sit near the allocated
figure above with shared GPU memory at zero. Anything in shared means part of
the model is being reached over PCIe, which costs roughly 5x on every call.

## Constrained label scoring

Each of the 8 candidate labels is appended to the prompt and its full token
sequence scored, then softmaxed over the 8 log-likelihoods. This makes
unmapped predictions impossible and yields a distribution usable for ECE.

Note the labels are not equal length: A2/B1/B2/C1 are 2 tokens and the plus
levels are 3. Summed log-probs therefore carry a mild length penalty on the
plus levels. Set `LENGTH_NORMALISE = True` to divide by token count if the
predictions look systematically skewed away from them.

In [4]:
# %%
LENGTH_NORMALISE = False

EOT_ID = tok.convert_tokens_to_ids("<turn|>")
assert EOT_ID is not None and EOT_ID >= 0, "check the eot token string"

# The bare labels are prefixes of one another: A2 = [236776, 236778] and
# A2+ = [236776, 236778, 236862]. Summed log-probs therefore made every plus
# level strictly lower than its base level, so they could never be predicted.
# Appending the end-of-turn token (which training targets also ended with)
# breaks the prefix relation and makes the comparison valid.
LABEL_TOKEN_IDS = {lvl: tok(lvl, add_special_tokens=False).input_ids + [EOT_ID]
                   for lvl in LEVELS}
LAB_IDS = [LABEL_TOKEN_IDS[lvl] for lvl in LEVELS]
MAX_LAB = max(len(l) for l in LAB_IDS)
print({k: len(v) for k, v in LABEL_TOKEN_IDS.items()})


@torch.no_grad()
def score_labels(text):
    """Return (8,) softmax distribution over LEVELS."""
    prompt_ids = tok.apply_chat_template(
        build_messages(text), tokenize=True, add_generation_prompt=True)
    start = len(prompt_ids)

    seqs = [prompt_ids + l for l in LAB_IDS]
    maxlen = max(len(s) for s in seqs)
    pad_id = tok.pad_token_id

    input_ids = torch.full((len(seqs), maxlen), pad_id, dtype=torch.long)
    attn = torch.zeros((len(seqs), maxlen), dtype=torch.long)
    for i, s in enumerate(seqs):
        input_ids[i, :len(s)] = torch.tensor(s)
        attn[i, :len(s)] = 1

    input_ids, attn = input_ids.to(model.device), attn.to(model.device)
    logits = model(input_ids=input_ids, attention_mask=attn).logits

    sliced = logits[:, start - 1: start - 1 + MAX_LAB, :]
    logprobs = torch.log_softmax(sliced.float(), dim=-1)

    ll = np.zeros(NUM_CLASSES)
    for i, l in enumerate(LAB_IDS):
        total = sum(logprobs[i, j, input_ids[i, start + j]].item()
                    for j in range(len(l)))
        ll[i] = total / len(l) if LENGTH_NORMALISE else total
    
    del logits, sliced, logprobs

    ll -= ll.max()
    e = np.exp(ll)
    return e / e.sum()

{'A2': 3, 'A2+': 4, 'B1': 3, 'B1+': 4, 'B2': 3, 'B2+': 4, 'C1': 3, 'C1+': 4}


### Timing check

Run this twice. The first call includes warmup; the second is the real
figure. Expect roughly 2-3s. If it is 10s or more, VRAM is spilling and the
full loop will crawl: restart the kernel rather than waiting it out.

In [5]:
import time

_t = time.time()
_p = score_labels(df[cfg.text_col].iloc[0])
print(f"{time.time() - _t:.2f}s  |  allocated %.1f GB | reserved %.1f GB" % vram())
print(dict(zip(LEVELS, _p.round(3))))
print("pred:", LEVELS[_p.argmax()], "| gold:", df[cfg.label_col].iloc[0])

3.25s  |  allocated 12.6 GB | reserved 16.2 GB
{'A2': 0.018, 'A2+': 0.13, 'B1': 0.517, 'B1+': 0.277, 'B2': 0.054, 'B2+': 0.003, 'C1': 0.001, 'C1+': 0.0}
pred: B1 | gold: B1


## Predict CEFR levels

The periodic `empty_cache()` stops fragmentation building up over the run.
Without it throughput degrades steadily as reserved memory creeps toward the
32GB ceiling.

In [6]:
# %%
# after repeated problems with the inference getting stuck on 422 texts, I split it into 2 runs
# for the first run
# START, END = 0, 400

# for the second run
START, END = 400, 599

chunk = []
for i, text in enumerate(tqdm(df[cfg.text_col].iloc[START:END])):
    chunk.append(score_labels(text))
    if i % cfg.empty_cache_every == 0:
        torch.cuda.empty_cache()

chunk = np.vstack(chunk)
out = f"{cfg.run_dir}/probs_{START}_{END}.npy"
np.save(out, chunk)
print(out, chunk.shape, "| reserved %.1f GB" % torch.cuda.memory_reserved(0) if False else "")
print("allocated %.1f GB | reserved %.1f GB" % vram())

  0%|          | 0/199 [00:00<?, ?it/s]

runs/gemma4_lora/probs_400_599.npy (199, 8) 
allocated 8.3 GB | reserved 28.4 GB


## Combine 2 runs together

In [7]:
# %%
probs = np.vstack([np.load(f"{cfg.run_dir}/probs_0_400.npy"),
                   np.load(f"{cfg.run_dir}/probs_400_599.npy")])
gold = df[cfg.label_col].map(LABEL2ID).to_numpy()
pred = probs.argmax(axis=1)
print(probs.shape, len(gold))

(599, 8) 599


## Metrics

In [8]:
m = evaluate_predictions(gold, pred)
print(json.dumps({k: round(float(v), 4) for k, v in m.items()}, indent=2))

{
  "qwk": 0.8549,
  "mae": 0.576,
  "accuracy": 0.5142,
  "adjacent_accuracy": 0.9232,
  "precision_macro": 0.3684,
  "recall_macro": 0.3888,
  "f1_macro": 0.368,
  "precision_weighted": 0.4805,
  "recall_weighted": 0.5142,
  "f1_weighted": 0.4824
}


### Prediction distribution

The prompting conditions showed central tendency bias, with predictions
compressed toward the middle levels. Compare the two columns below.

In [9]:
dist = pd.DataFrame({
    "predicted": pd.Series([LEVELS[p] for p in pred]).value_counts(),
    "gold": pd.Series([LEVELS[g] for g in gold]).value_counts(),
}).reindex(LEVELS).fillna(0).astype(int)
print(dist)

     predicted  gold
A2           0    30
A2+        195   136
B1          78   128
B1+        129    95
B2         122   110
B2+         31    49
C1          44    37
C1+          0    14


## Confusion matrix

In [10]:
# %%
import numpy as np
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

counts = confusion_matrix(gold, pred, labels=list(range(NUM_CLASSES)))
row_sums = counts.sum(axis=1, keepdims=True)
with np.errstate(divide="ignore", invalid="ignore"):
    norm = np.nan_to_num(np.divide(counts, row_sums, where=row_sums != 0))

annot = np.empty_like(counts, dtype=object)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        annot[i, j] = (f"{norm[i, j] * 100:.1f}%<br>({counts[i, j]})"
                       if counts[i, j] else "")

fig = go.Figure(go.Heatmap(
    z=norm, x=LEVELS, y=LEVELS,
    text=annot, texttemplate="%{text}",
    hoverongaps=False,
    colorscale="Greys", zmin=0, zmax=1,
    colorbar=dict(title="Row-normalized"),
))
fig.update_xaxes(title_text="Predicted")
fig.update_yaxes(title_text="True", autorange="reversed")
fig.update_layout(
    title=dict(text=f"Gemma 4 12B LoRA — {SPLIT}"),
    font=dict(family="Arial", size=16, color="black"),
    width=620, height=560,
    margin=dict(l=80, r=100, t=100, b=80),
)
fig.show()

## Predictions CSV

Same column layout as the encoder runs, so the ECE and AUROC cells in the
PROBABILITIES notebooks can read this directly.

In [11]:
out = pd.DataFrame({
    "gold": gold,
    "pred": pred,
    "gold_label": [LEVELS[g] for g in gold],
    "pred_label": [LEVELS[p] for p in pred],
    "correct": (gold == pred).astype(int),
    "adjacent": (np.abs(gold - pred) <= 1).astype(int),
    "p_pred": probs[np.arange(len(pred)), pred],
})
for k in range(NUM_CLASSES):
    out[f"p_{LEVELS[k]}"] = probs[:, k]

path = f"{cfg.run_dir}/{SPLIT}_predictions.csv"
out.to_csv(path, index=False)
print(path)
print(out[["correct", "adjacent", "p_pred"]].mean().round(3).to_dict())

runs/gemma4_lora/val_predictions.csv
{'correct': 0.514, 'adjacent': 0.923, 'p_pred': 0.461}


In [12]:
# %%
import pandas as pd
from sklearn.metrics import roc_auc_score

srt = np.sort(probs, axis=1)[:, ::-1]
rows = []
for i, (g, p) in enumerate(zip(gold, pred)):
    g, p = int(g), int(p)
    lo, hi = max(0, p - 1), min(NUM_CLASSES - 1, p + 1)
    rows.append(dict(
        gold=g, pred=p, gold_label=LEVELS[g], pred_label=LEVELS[p],
        correct=int(g == p), adjacent=int(abs(g - p) <= 1),
        p_pred=float(probs[i, p]),
        p_adjacent=float(probs[i, lo:hi + 1].sum()),
        margin=float(srt[i, 0] - srt[i, 1]),
        entropy=float(-(probs[i] * np.log(probs[i] + 1e-12)).sum()),
        exp_level=float((probs[i] * np.arange(NUM_CLASSES)).sum()),
        **{f"p_{lvl}": float(probs[i, j]) for j, lvl in enumerate(LEVELS)},
    ))

dev_df = pd.DataFrame(rows)
dev_df.to_csv(f"{cfg.run_dir}/{SPLIT}_predictions.csv", index=False)


def ece(d, n_bins=10):
    b = pd.cut(d["p_pred"], np.linspace(0, 1, n_bins + 1))
    g = d.groupby(b, observed=True)
    gap = (g["correct"].mean() - g["p_pred"].mean()).abs()
    return float((gap * g.size()).sum() / len(d))


print(f"mean p_pred  {dev_df['p_pred'].mean():.3f}")
print(f"mean entropy {dev_df['entropy'].mean():.3f}  (max {np.log(NUM_CLASSES):.3f})")
print(f"ECE          {ece(dev_df):.4f}")
print()
for col, s in [("p_pred", dev_df["p_pred"]), ("margin", dev_df["margin"]),
               ("entropy", -dev_df["entropy"]), ("p_adjacent", dev_df["p_adjacent"])]:
    print(f"{col:12s} exact {roc_auc_score(dev_df['correct'], s):.3f}   "
          f"adjacent {roc_auc_score(dev_df['adjacent'], s):.3f}")
print()
print(dev_df.groupby("correct")[["p_pred", "margin", "entropy"]].mean().round(3))

mean p_pred  0.461
mean entropy 1.247  (max 2.079)
ECE          0.0646

p_pred       exact 0.584   adjacent 0.762
margin       exact 0.589   adjacent 0.749
entropy      exact 0.575   adjacent 0.750
p_adjacent   exact 0.578   adjacent 0.730

         p_pred  margin  entropy
correct                         
0         0.445   0.138    1.274
1         0.476   0.178    1.221


In [13]:
# %%
BANDS = [(0.70, 1.01, "> .70"), (0.60, 0.70, ".60–.70"), (0.50, 0.60, ".50–.60"),
         (0.40, 0.50, ".40–.50"), (0.30, 0.40, ".30–.40"), (0.00, 0.30, "≤ .30")]

n = len(dev_df)
rows = []
for lo, hi, name in BANDS:
    s = dev_df[(dev_df["p_pred"] >= lo) & (dev_df["p_pred"] < hi)]
    rows.append(dict(
        conf=name,
        preds=len(s),
        share=f"{len(s)/n*100:.0f}%",
        acc=round(s["correct"].mean(), 2) if len(s) else None,
        adj=round(s["adjacent"].mean(), 2) if len(s) else None,
    ))

band_df = pd.DataFrame(rows)
print(band_df.to_string(index=False))

   conf  preds share  acc  adj
  > .70      0    0%  NaN  NaN
.60–.70     89   15% 0.60 1.00
.50–.60    129   22% 0.59 0.98
.40–.50    187   31% 0.52 0.93
.30–.40    149   25% 0.43 0.83
  ≤ .30     45    8% 0.40 0.87
